# 17.10 壓縮後怎麼實際運算？


低位元檔案已變小，生成速度卻沒有變快，甚至更慢，可能是什麼原因？因為「怎麼存」與「怎麼算」是兩個問題。參考QuantizedLinear每次先拆碼、乘scale，重建浮點權重，再交給普通線性運算；它能提供可核對的數值基準，但多了還原工作和臨時矩陣。前置是[只量化權重](https://birdhackor.github.io/tiny-perceptron-vlm/17.7.html)與[實際打包](https://birdhackor.github.io/tiny-perceptron-vlm/17.8.html)。

例如一層有32個輸出、64個輸入，權重2048個FP32占8192bytes，bias有32個占128bytes，原始數字合計8320。四位元權重打包占1024bytes，各輸出列的scale占128bytes，bias仍128，共1280。儲存明顯減少，然而參考forward還原出的浮點權重又是8192bytes，不會因來源小就自動變成低位元乘法。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from tiny_perceptron.quantization import QuantizedLinear

layer = nn.Linear(64, 32)
compressed = QuantizedLinear(layer, bits=4)
x = torch.ones(2, 64)
float_bytes = sum(p.numel() * p.element_size() for p in layer.parameters())
print("原層數字bytes", float_bytes)
print("壓縮buffer bytes", compressed.storage_bytes())
print("整數碼bytes", compressed.values.numel() * compressed.values.element_size())
print("運算結果", tuple(compressed(x).shape), compressed(x).dtype)

前三行應為8320、1280與1024。第一行含原權重與bias，第二行含壓縮碼、scale與bias，第三行只看碼。這種分開列報讓你看見刻度與偏移的固定成本，而不把「8320變1024」當作完整層的儲存比。輸入兩列64項特徵，最後輸出形狀 `(2,32)`、型別torch.float32：數值運算仍為浮點。

專用低位元kernel指理解打包格式與刻度、替特定硬體安排乘法和讀取的底層運算。有的邊讀小塊權重邊反量化，不建立整張浮點權重；有的使用特定整數或混合精度路徑。能否加速依硬體、形狀、位元格式與實作而定，不是把buffer改成uint8就能讓普通Linear知道怎麼算。

因此完整評估至少有四件不同的量：檔案bytes、載入時的最高記憶體、運行時的最高記憶體，以及[prefill與decode耗時](https://birdhackor.github.io/tiny-perceptron-vlm/16.2.html)。decode可能受讀權重頻寬影響，小型CPU矩陣則可能被拆碼開銷主導。每次還原的暫存資料也算進最高占用，不能只用模塊buffer大小當成執行需求。

本節不加入新的硬體套件，先保留可讀的參考結果；選擇其他後端時，需要核對支援格式、數值與同一輸入下的計時。量速度前先暖機，也就是載入模型後，用同樣形狀的輸入試跑幾次，這些試跑不列入正式計時。若後端會在第一次呼叫編譯運算，先等這次呼叫完成，再開始重複測量並取平均；載入與首次編譯耗時另外記錄。如何計時可看[5.9的小型測量](https://birdhackor.github.io/tiny-perceptron-vlm/5.9.html)。

練習只把bits改成8。先預測整數碼2048bytes，scale與bias各128，總buffer2304，輸出仍float32、形狀仍 `(2,32)`，再執行核對。兩版存儲不同，運算介面的形狀相同；速度必須另量，不能從這些bytes代算。
